In [ ]:
import flow
import numpy as np
import pandas as pd
import os
import sys
import subprocess
import shutil
import matplotlib.pyplot as plt
from flow import FlowProject, directives
from pymser import pymser
import warnings
import panedr
from block_average.block_average import block_average

In [ ]:
#Maxwell distribution ex plots. Not used in final figs.

import MDAnalysis as mda
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt

# ---------- User inputs ----------
#042907986f66368d7fd5e2536a646d49 Gly 293.15

#1dbe7211624e244c4321b6f329f54cba DMSO 323.15
#66c77ba71d27b31e15f416cab75cbf78 DMSO 333.15
mode = "vap"
to_dir = "Build_GPs/vle_iters/workspace/042907986f66368d7fd5e2536a646d49/inter_prod/"
from scipy.stats import maxwell

topology = "Build_GPs/vle_iters/workspace/042907986f66368d7fd5e2536a646d49/inter_prod/inter_prod.tpr"
trajectory = "Build_GPs/vle_iters/workspace/042907986f66368d7fd5e2536a646d49/inter_prod/inter_prod.trr"
if mode == "liq":
    zmin = 16.1863   # nm
    zmax = 31.186    # nm
else:
    zmin = 15.8066
    zmax = 31.6606   # nm
output_csv = f"ke_atom_{mode}.csv"
# per_molecule_csv = "ke_per_molecule_by_frame.csv"   # optional; large files possible

# ---------- constants ----------
AMU_TO_KG = 1.66053906660e-27        # kg per atomic mass unit
AVOGADRO = 6.02214076e23             # mol^-1
KB = 1.380649e-23                    # J/K (not used but kept for reference)

# ---------- load universe ----------
u = mda.Universe(topology, trajectory)

# sanity: velocities present?
if not hasattr(u.atoms, "velocities") or u.atoms.velocities is None:
    # for some formats MDAnalysis only exposes velocities per frame via ts, but for GROMACS trr it should be available
    print("Warning: atom velocities are not available globally. We'll check frame by frame.")

# ---------- prepare residue/molecule mapping ----------
# We'll treat each residue as a molecule. Build list of residue atom indices (integer positions into atoms array)
residues = list(u.residues)   # MDAnalysis Residue objects in topology order
n_res = len(residues)
print(f"Found {n_res} residues")

# For speed, precompute atom indices per residue and per-atom masses in kg
atom_masses_amu = u.atoms.masses                                # shape (N,) in amu
atom_masses_kg = atom_masses_amu * AMU_TO_KG                    # shape (N,)

# ---------- main loop over frames ----------
rows = []             # store per-frame summary
rows_per_mol = []     # optional per-residue per-frame entries

count = 0

print(len(u.trajectory))
# We'll iterate frames; use tqdm for progress if available
for ts in tqdm(u.trajectory, desc="Frames"):
    time_ps = ts.time             # time in ps
    frame_index = ts.frame       # integer frame index

    # get all velocities for atoms in this frame
    # MDAnalysis provides u.atoms.velocities for current ts (units depend on reader, with GROMACS usually nm/ps)
    positions_nm = mda.units.convert(u.atoms.positions.copy(), "Angstrom", "nm")      # shape (N,3) in nm
    velocities = u.atoms.velocities.copy()   # shape (N,3) in A/ps

    # convert velocities to m/s: 1 nm/ps = 1000 m/s
    vel_m_s = mda.units.convert(velocities, "Angstrom/ps", "m/s")   # shape (N,3)

    # compute per-atom squared speed
    v2 = np.sum(vel_m_s * vel_m_s, axis=1)         # shape (N,)

    # compute per-atom kinetic energy (J): 0.5 * m * v^2
    ke_atom_J = 0.5 * atom_masses_kg * v2         # shape (N,)

    # compute per-residue center-of-mass z coordinate to decide selection
    # Use residue.atoms.center_of_mass() which returns nm if positions are in nm (MDAnalysis uses same units as reader)
    # To be safe treat COM z in nm because we are comparing to zmin/zmax in nm
    if mode == "liq":
        mask = (positions_nm[:, 2] >= zmin) & (positions_nm[:, 2] <= zmax)
    else:
        mask = (positions_nm[:, 2] <= zmin) | (positions_nm[:, 2] >= zmax)
    sel_residue_indices = np.nonzero(mask)[0]

    ke_selected = ke_atom_J[sel_residue_indices]
    Temp_sel = np.mean((2.0 / 3.0) * (ke_selected / KB)) # in Kelvin
    total_ke_J = ke_selected.sum()

    # If none selected, store zeros and continue
    if len(sel_residue_indices) == 0:
        rows.append({
            "frame": frame_index,
            "time_ps": time_ps,
            "n_selected_molecules": 0,
            "total_ke_J": 0.0,
            "avg temperature_K": 0.0
        })
        # still optionally record per-molecule rows (empty for none)
        continue

    rows.append({
        "frame": frame_index,
        "time_ps": time_ps,
        "n_selected_molecules": len(sel_residue_indices),
        "total_ke_J": total_ke_J,
        "avg temperature_K": Temp_sel,
    })


    if count == 0:
        if mode == "liq":
            name = "Liquid"
        else:
            name = "Vapor"

            # Maxwell-Boltzmann KE distribution
        def maxwell_ke_pdf(E, T):
            return (2/np.sqrt(np.pi)) * (1/(KB*T)**1.5) * np.sqrt(E) * np.exp(-E/(KB*T))

        # Known system temperature (in Kelvin)
        T = 293.15   # <-- set your temperature here
        #Print a histogram of the energies
        counts, bins, patches  = plt.hist(ke_selected, bins=40, edgecolor='black', density=True, label = "MD")
        E_vals = np.linspace(bins[0], bins[-1], 400)
        plt.plot(E_vals, maxwell_ke_pdf(E_vals, T), 'r-', lw=2, label=f'Maxwell-Boltzmann (T={T:.0f} K)')
        # Plot the theoretical distribution
        plt.legend()
        plt.xlabel('Kinetic Energy (Joules)')
        plt.ylabel('Probablility Density')
        plt.title(f'Histogram of KE of {name} Atoms')
        plt.grid(True)
        plt.savefig(to_dir + f'ke_atom_hist_{mode}.png')
        count += 1

# ---------- save outputs ----------

df_summary = pd.DataFrame(rows)
df_summary.to_csv(to_dir+output_csv, index=False)
print(f"Wrote per-frame summary to {to_dir+output_csv}")

In [ ]:
import MDAnalysis as mda
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
#Maxwell distribution ex plots. Not used in final figs.

# ---------- User inputs ----------
mode = "vap"
topology = "Build_GPs/vle_iters/workspace/042907986f66368d7fd5e2536a646d49/inter_prod/inter_prod.tpr"
trajectory = "Build_GPs/vle_iters/workspace/042907986f66368d7fd5e2536a646d49/inter_prod/inter_prod.trr"
zmin = 16.1863   # nm
zmax = 31.186    # nm
output_csv = f"mol_ke_{mode}.csv"

# ---------- constants ----------
AMU_TO_KG = 1.66053906660e-27        # kg per atomic mass unit
AVOGADRO = 6.02214076e23             # mol^-1
KB = 1.380649e-23                    # J/K (not used but kept for reference)

# ---------- load universe ----------
u = mda.Universe(topology, trajectory)

if mode == "liq":
    zmin = 16.1863   # nm
    zmax = 31.186    # nm
else:
    zmin = 15.8066
    zmax = 31.6606   # nm

# ---------- prepare residue/molecule mapping ----------
# We'll treat each residue as a molecule. Build list of residue atom indices (integer positions into atoms array)
residues = list(u.residues)   # MDAnalysis Residue objects in topology order
n_res = len(residues)
print(f"Found {n_res} residues")

# For speed, precompute atom indices per residue and per-atom masses in kg
res_atom_indices = [res.atoms.indices for res in residues]     # numpy arrays, 0-based indices into Universe.atoms
atom_masses_amu = u.atoms.masses                                # shape (N,) in amu
atom_masses_kg = atom_masses_amu * AMU_TO_KG                    # shape (N,)

# ---------- main loop over frames ----------
rows = []             # store per-frame summary
rows_per_mol = []     # optional per-residue per-frame entries

count = 0
# We'll iterate frames; use tqdm for progress if available
for ts in tqdm(u.trajectory, desc="Frames"):
    time_ps = ts.time             # time in ps
    frame_index = ts.frame       # integer frame index

    # get all velocities for atoms in this frame
    # MDAnalysis provides u.atoms.velocities for current ts (units depend on reader, with GROMACS usually nm/ps)
    velocities = u.atoms.velocities.copy()   # shape (N,3) in nm/ps

    # convert velocities to m/s: 1 nm/ps = 1000 m/s
    vel_m_s = mda.units.convert(velocities, "Angstrom/ps", "m/s")   # shape (N,3)

    # compute per-atom squared speed
    v2 = np.sum(vel_m_s * vel_m_s, axis=1)         # shape (N,)

    # compute per-atom kinetic energy (J): 0.5 * m * v^2
    ke_atom_J = 0.5 * atom_masses_kg * v2         # shape (N,)

    # compute per-residue center-of-mass z coordinate to decide selection
    # Use residue.atoms.center_of_mass() which returns nm if positions are in nm (MDAnalysis uses same units as reader)
    # To be safe treat COM z in nm because we are comparing to zmin/zmax in nm
    sel_residue_indices = []
    for i, res in enumerate(residues):
        com = mda.units.convert(res.atoms.center_of_mass(), "Angstrom", "nm")   # array length 3, units same as positions (GROMACS: nm)
        com_z = com[2]
        if mode == "liq":
            if (com_z >= zmin) and (com_z <= zmax):
                sel_residue_indices.append(i)
        else:
            if (com_z <= zmin) or (com_z >= zmax):
                sel_residue_indices.append(i)

    # If none selected, store zeros and continue
    if len(sel_residue_indices) == 0:
        rows.append({
            "frame": frame_index,
            "time_ps": time_ps,
            "n_selected_molecules": 0,
            "total_ke_J": 0.0,
            "total_ke_kJ_per_mol": 0.0
        })
        # still optionally record per-molecule rows (empty for none)
        continue

    # Sum KE per selected residue (molecule)
    ke_per_residue_J = np.zeros(len(sel_residue_indices), dtype=float)
    for j_idx, res_i in enumerate(sel_residue_indices):
        atom_idx = res_atom_indices[res_i]   # numpy array of atom indices
        ke_per_residue_J[j_idx] = np.sum(ke_atom_J[atom_idx])

    n_atoms_per_mol = len(res_atom_indices[0])
    n_atoms_phase = len(sel_residue_indices) * n_atoms_per_mol

    # total KE for the selected set (J)
    total_ke_J = float(np.sum(ke_per_residue_J))
    temps_sel = (2.0 / 3.0) * (ke_per_residue_J / (n_atoms_per_mol * KB)) # in Kelvin

    rows.append({
        "frame": frame_index,
        "time_ps": time_ps,
        "n_selected_molecules": len(sel_residue_indices),
        "total_ke_J": total_ke_J,
        "avg temperature_K": np.mean(temps_sel)
    })

    if count == 200:
        if mode == "liq":
            name = "Liquid"
        else:
            name = "Vapor"

        #Print a histogram of the energies
        counts, bins, patches  = plt.hist(ke_per_residue_J, bins=40, edgecolor='black', density=True)

        E_vals = np.linspace(bins[0], bins[-1], 400)
        plt.xlabel('Kinetic Energy (Joules)')
        plt.ylabel('Probablility Density')
        plt.title(f'Histogram of KE of {name} Molecules')
        plt.grid(True)
        plt.savefig(to_dir + f'ke_mol_hist_{mode}.png')
    count += 1

# ---------- save outputs ----------
df_summary = pd.DataFrame(rows)
df_summary.to_csv(to_dir + output_csv, index=False)
print(f"Wrote per-frame summary to {to_dir + output_csv}")

In [ ]:
def calc_mass_dens(density, mode = "liq"):
    #Find the region attributed to liquid density
    # fraction = 0.4
    # n_shift = int(len(density[:, 1]) * fraction)
    # # Circular shift
    # density[:, 1] = np.roll(density[:, 1], -n_shift)
    
    mass_dens_z = density[:, 1]
    find_liq_slab = find_bulk_liq_index(density, mode = mode)
    range_for_liq_dens = find_liq_slab[0]
    range_org_liq = find_liq_slab[1]
    median_dens_liq = find_liq_slab[3]
    #Calculate the density
    prop_vals = mass_dens_z[range_for_liq_dens] #kg/m^3
    fig, ax = plt.subplots()

    ax.scatter(density[:, 0], density[:, 1], label='Density', color='blue')
    ax.axhline(y=median_dens_liq, color='g', linestyle='--', label='Median Bulk Liq. Density')

    ax.set_xlabel('Z (nm)')
    ax.set_ylabel('Density (kg/m^3)')

    med_dens_pt = find_liq_slab[4]

    ax.scatter(density[range_for_liq_dens, 0],
            density[range_for_liq_dens, 1],
            label='Bulk Liquid',
            color='red')

    # ax.scatter(density[range_org_liq, 0], density[range_org_liq, 1], color='green')

    # ax.scatter(density[med_dens_pt, 0],
    #         density[med_dens_pt, 1],
    #         label='Median Bulk Liq. Density',
    #         color='orange')

    ax.legend(loc='lower center',
          bbox_to_anchor=(0.5, 1.02),
          ncol=3,
          frameon=False)
    print(np.mean(prop_vals))
    print("Liq Dens Range")
    # plt.legend()
    print(max(density[range_for_liq_dens, 0]), min(density[range_for_liq_dens, 0]))
    return prop_vals, fig
from scipy.signal import find_peaks
from findpeaks import findpeaks


def find_bulk_liq_index(density, mode = "liq"):
    #Use np.diff to approximate the 1st derivative
    ES_numdens_z = density[:,1]
    x = density[:,0]
    # dy = np.diff(ES_numdens_z)
    dy = np.gradient(ES_numdens_z, x)
    # dy = np.gradient(np.gradient(ES_numdens_z, x), x)
    # print(dy)
    # plt.scatter(np.arange(0,len(dy),1), dy, label='ES', color='blue')
    # plt.scatter(results['x'].iloc[interfaces], results['y'].iloc[interfaces], label='ES', color='red')
    #Use findpeaks to find the peaks and valleys, interpolating to get as close as possible
    fp = findpeaks(lookahead=1, interpolate=10, verbose=0)
    results = fp.fit(dy)["df_interp"]

    all_peaks = results[results['peak'] | results['valley']].index.values
    #get the highest peak and the lowest valley, these are the interfaces
    y_vals = results['y'].iloc[all_peaks]
    peak_index = all_peaks[np.argmax(y_vals)]
    valley_index = all_peaks[np.argmin(y_vals)]

    interfaces = [peak_index, valley_index]

    #Divide the indices by 10 to get the correct index for the density based on interpolation
    interfaces = [int(i/10) for i in interfaces]

    #Get the range of indices for the bulk liquid density regardless of whether interfaces have shifted
    if interfaces[0] < interfaces[1]:
        range_org_liq = list(range(interfaces[0], interfaces[1] + 1))
    else:
        range_org_liq = list(range(interfaces[0], len(results)//10)) + list(range(0, interfaces[1] + 1))

    #Get the indecies of the vapor range (the opposite of range_org_liq)
    if interfaces[0] < interfaces[1]:
        range_org_vapor = list(range(interfaces[1] + 1, len(results)//10)) + list(range(0, interfaces[0]))
    else:
        range_org_vapor = list(range(interfaces[1] + 1, interfaces[0]))

    if_r = [peak_index, valley_index]

    # if if_r[0] < if_r[1]:
    #     range_org_liq1 = list(range(if_r[0], if_r[1] + 1))
    # else:
    #     range_org_liq1 = list(range(if_r[0], len(results))) + list(range(0, if_r[1] + 1))

    # plt.scatter(results['x']*max(x)/(len(x)*10), results['y'], label='ES', color='blue')
    # plt.scatter(results['x'].iloc[all_peaks]*max(x)/(len(x)*10), results['y'].iloc[all_peaks], label='ES', color='red')
    # plt.scatter(results['x'].iloc[if_r[0]]*max(x)/(len(x)*10), results['y'].iloc[if_r[0]], label='ES', color='green')
    # plt.scatter(results['x'].iloc[if_r[1]]*max(x)/(len(x)*10), results['y'].iloc[if_r[1]], label='ES', color='green')
    
    # plt.xlabel('Z (nm)')
    # plt.ylabel('d_rho/dz (kg/m^3/nm)')
    # plt.show()

    # #Divide the indices by 10 to get the correct index for the density based on interpolation
    # interfaces = [int(i/10) for i in if_r]
    # range_org_liq = [int(i/10) for i in range_org_liq1]
    median_dens_liq = np.median(ES_numdens_z[range_org_liq])
    differences = np.abs(ES_numdens_z - median_dens_liq)
    med_dens_pt = np.argmin(differences)
    # plt.scatter(results['x']*max(x)/(len(x)*10), results['y'], label='ES', color='blue')
    # plt.scatter(results['x'].iloc[range_org_liq*10]*max(x)/(len(x)*10), results['y'].iloc[range_org_liq*10], label='ES', color='red')

    median_dens_vap = np.mean(ES_numdens_z[range_org_vapor])
    differences_vap = np.abs(ES_numdens_z - median_dens_vap)
    med_dens_pt_vap = np.argmin(differences_vap)

    # plt.xlabel('Z (nm)')
    # plt.ylabel('d_rho/dz (kg/m^3/nm)')
    # plt.show()

    # Find the first and last index where density >= median
    valid_idx = np.where(ES_numdens_z[range_org_liq] >= median_dens_liq)[0]
    if valid_idx.size > 0:
        start = valid_idx[0]
        end = valid_idx[-1] + 1  # +1 to include the last valid index
        range_for_liq_dens = range_org_liq[start:end]

        # full_range = range_org_liq[start:end]
        # print(len(range_org_liq), len(full_range), len(range_org_liq)/len(full_range))
        # cutoff = int(len(full_range) * 0.85)
        # trim = (len(full_range) - cutoff) // 2
        # range_for_liq_dens = full_range[trim:trim + cutoff]

    else:
        range_for_liq_dens = np.array([], dtype=int)

    median_dens_liq = np.median(ES_numdens_z[range_for_liq_dens])
    differences = np.abs(ES_numdens_z - median_dens_liq)

    #Do the same for vapor
    valid_idx = np.where(ES_numdens_z[range_org_vapor] <= median_dens_vap)[0]
    # print(valid_idx)
    if valid_idx.size > 0:
        start = valid_idx[0]
        end = valid_idx[-1] + 1 # +1 to include the last valid index
        range_for_vapor_dens = range_org_vapor[start:end]
    else:
        range_for_vapor_dens = np.array([], dtype=int)

    median_dens_vap = np.mean(ES_numdens_z[range_for_vapor_dens])
    differences_vap = np.abs(ES_numdens_z - median_dens_vap)
    print("Vap Dens Range")
    idx1_vap = range_org_vapor[0]
    idx2_vap = range_org_vapor[-1]
    print(density[idx1_vap, 0], density[idx2_vap, 0])
    # med_dens_pt = np.argmin(differences)
    # print(range_org_liq)
    # range_for_liq_dens = range_org_liq
    # print(x[max(range_for_liq_dens)], x[min(range_for_liq_dens)])

    if mode == "liq":
        return range_for_liq_dens, range_org_liq, ES_numdens_z, median_dens_liq, med_dens_pt
    elif mode == "vap":
        return range_for_vapor_dens, range_org_vapor, ES_numdens_z, median_dens_vap, med_dens_pt_vap


In [ ]:
#Make IFT box interface validation figs
import glob
#for all IFT things
#Get all files that match a certain pattern
files = sorted(glob.glob("Build_GPs/vle_iters/workspace/*/calc_props/ift_liq_dens.xvg"))
files2 = sorted(glob.glob("Opt_ES/ift_val_opt/workspace/*/calc_props/ift_liq_dens.xvg"))
files3 = sorted(glob.glob("Opt_ES/ift_val_no_opt/workspace/*/calc_props/ift_liq_dens.xvg"))
all_files = files + files2 + files3
for file in all_files:
    try:
        density = np.loadtxt(file, comments=["#", "@"]) #Gly weird
        prop_vals, fig = calc_mass_dens(density, "liq")
        save_fig_loc = file.replace("calc_props/ift_liq_dens.xvg", "calc_props/ift_prod_dens.png")
        #Save the figure to Build_GPs/vle_iters/workspace/042907986f66368d7fd5e2536a646d49/calc_props
        fig.savefig(save_fig_loc, dpi=300, bbox_inches='tight')
    except:
        pass